# Batch correction with scANVI (semi-supervised)

scANVI (Xu et al., *Mol Syst Biol* 2021) extends scVI with a supervised classifier head trained on partially-labelled cells. Cells without a confident label get a placeholder value (e.g. `'Unknown'`); scANVI both batch-corrects and predicts labels for the unknowns. Best when you have a high-confidence reference dataset and want to transfer labels to a query.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install scvi-tools`.

**Input contract**: raw counts in `adata.layers['counts']` + a `labels_key` obs column with the cell-type label per cell. Cells whose label equals `unlabeled_category` (default `'Unknown'`) are predicted by the model. The predictions are written to `adata.obs['scANVI_predicted_labels']`.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='scANVI')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
model = ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='scANVI',
    # Required (no default — scANVI is semi-supervised):
    labels_key='celltype',
    unlabeled_category='Unknown',
    # Architecture:
    n_latent=20,
    # Optimisation:
    max_epochs=200, early_stopping=True,
)
model

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **scANVI (semi-supervised)** it is `adata.obsm['X_scANVI']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_scanvi'] = ov.utils.mde(adata.obsm['X_scANVI'])
ov.pl.embedding(
    adata,
    basis='X_mde_scanvi',
    color=['batch'],
    frameon='small',
    title='scANVI (semi-supervised) — coloured by batch',
)

## Key parameters

**Required**:
- `labels_key` — obs column with cell-type labels.
- `unlabeled_category` — value in `labels_key` that means "predict this". Default `'Unknown'`.

**Architecture / optimisation**: identical to scVI — see the scVI zoo entry for the full list.

**Outputs**:
- `adata.obsm['X_scANVI']` — batch-corrected latent.
- `adata.obs['scANVI_predicted_labels']` — predicted labels for unlabeled cells (best-effort).


## Related tutorials

- `scVI` — fully unsupervised version when you have no labels.
- `scPoli` — alternative semi-supervised approach with per-condition prototypes.

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).